In [2]:
import pandas as pd
import requests
import json
import os
import uuid
import hashlib
import logging
from datetime import datetime, date, timezone
from decimal import Decimal, ROUND_DOWN
import xml.etree.ElementTree as ET
from google.cloud import bigquery

In [15]:
exchange_rates_url = "https://api.exchangeratesapi.io/latest"  
exchange_rates_api_key = 'e09d4c9bbb3a9d7c1e04ec7b8c91052a'  #os.getenv("EXCHANGE_RATES_API_KEY")
ecb_url = "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-daily.xml"
gcp_project = "dataengineering-466011"
bq_dataset = "lexis_nexis_task"
bq_landing_table = "exchange_rates_landing"
bq_final_table = "exchange_rates"
job_id = str(uuid.uuid4())
logging.basicConfig(
    filename="exchange_rates_ingest.log",
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s: %(message)s"
)
logger = logging.getLogger("exchange_rates_ingest")

In [ ]:
from google.cloud import secretmanager

def get_secret(secret_id: str, project_id: str) -> str:
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{project_id}/secrets/{secret_id}/versions/latest"
    response = client.access_secret_version(request={"name": name})
    return response.payload.data.decode("UTF-8")

# Usage
exchange_rates_api_key = get_secret("ExchangeRatesAPIKey", gcp_project)
print(exchange_rates_api_key) 

e09d4c9bbb3a9d7c1e04ec7b8c91052a


In [ ]:
r = requests.get(url=exchange_rates_url, params={"access_key": exchange_rates_api_key})
#data = 
print(r.json())
raw_json = json.dumps(r.json(), sort_keys=True)
raw_json
hashlib.sha256(raw_json.encode("utf-8")).hexdigest()

logging.basicConfig(
    filename="exchange_rates_ingest.log",
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s: %(message)s"
)

#print(data)
#df = pd.DataFrame(data['rates'], index=[0])
#rint(df)

{'success': True, 'timestamp': 1757969347, 'base': 'EUR', 'date': '2025-09-15', 'rates': {'AED': 4.320112, 'AFN': 81.157208, 'ALL': 96.91756, 'AMD': 450.625771, 'ANG': 2.105869, 'AOA': 1078.573392, 'ARS': 1724.31436, 'AUD': 1.763284, 'AWG': 2.117157, 'AZN': 1.984099, 'BAM': 1.956638, 'BBD': 2.368114, 'BDT': 143.120045, 'BGN': 1.954442, 'BHD': 0.443479, 'BIF': 3463.904486, 'BMD': 1.176198, 'BND': 1.506845, 'BOB': 8.142485, 'BRL': 6.25879, 'BSD': 1.175803, 'BTC': 1.0202174e-05, 'BTN': 103.626357, 'BWP': 16.61011, 'BYN': 3.981304, 'BYR': 23053.489961, 'BZD': 2.364712, 'CAD': 1.620519, 'CDF': 3361.575483, 'CHF': 0.934613, 'CLF': 0.028538, 'CLP': 1119.528659, 'CNY': 8.373352, 'CNH': 8.373104, 'COP': 4595.995511, 'CRC': 592.253512, 'CUC': 1.176198, 'CUP': 31.169259, 'CVE': 110.827266, 'CZK': 24.310833, 'DJF': 209.033562, 'DKK': 7.465067, 'DOP': 74.041885, 'DZD': 152.471672, 'EGP': 56.660418, 'ERN': 17.642977, 'ETB': 169.256174, 'EUR': 1, 'FJD': 2.629038, 'FKP': 0.868014, 'GBP': 0.864806, 'GE

'4fa9b8e766509ec1f76e53d754d95b2f66f312dc6ca7d350fefb1afb9dba15d1'

In [4]:
def parse_exchangeratesapi(job_id: str) -> pd.DataFrame:
    try:
        r = requests.get(url=exchange_rates_url, params={"access_key": exchange_rates_api_key})
        r.raise_for_status()  # Raise an error for HTTP errors
    except requests.RequestException as e:
        print(f"Error fetching exchange rates: {e}")
        return pd.DataFrame()  # Return an empty DataFrame on error

    payload = r.json() # The JSON response from the API
    raw_json = json.dumps(payload, sort_keys=True) # Ensure consistent ordering
    raw_hash = hashlib.sha256(raw_json.encode("utf-8")).hexdigest() # Hash of the raw JSON payload
    rates = payload.get("rates", {}) # Dictionary of currency rates
    rate_date = payload.get("date")
    rows = []
    for quote, r in rates.items():  # Iterate over each currency and its rate
        rows.append({
            "record_id": str(uuid.uuid4()),
            "source": "ExchangeRates_API",
            "base_currency": 'EUR',
            "quote_currency": quote,
            "rate": float(r),
            "rate_date": pd.to_datetime(rate_date).date(),
            "retrieved_at": datetime.utcnow().isoformat(),
            "raw_hash": raw_hash,
            "raw_payload": raw_json,
            "ingest_job_id": job_id,
            "created_at": datetime.utcnow().isoformat()
        })
    return pd.DataFrame(rows)

df = parse_exchangeratesapi(job_id)

In [5]:
def parse_ecb_api(job_id: str) -> pd.DataFrame:
    try:
        r = requests.get(ecb_url, timeout=30)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"Error fetching ECB data: {e}")
        return pd.DataFrame()
    ecb_data = r.text
    root = ET.fromstring(ecb_data) # Parse the XML
    raw_hash = hashlib.sha256(ecb_data.encode("utf-8")).hexdigest() # Hash of the raw XML payload
    rows = []
    # find Cube elements with time attr
    # namespace handling
    ns = {'ns': 'http://www.ecb.int/vocabulary/2002-08-01/eurofxref'}
    for cube_time in root.findall('.//{http://www.ecb.int/vocabulary/2002-08-01/eurofxref}Cube[@time]'):
        time_attr = cube_time.attrib['time']
        for cube in cube_time.findall('{http://www.ecb.int/vocabulary/2002-08-01/eurofxref}Cube'):
            cur = cube.attrib.get('currency')
            rate_str = cube.attrib.get('rate')
            if not (cur and rate_str):
                continue
            rows.append({
                "record_id": str(uuid.uuid4()),
                "source": "ECB_API",
                "base_currency": "EUR",
                "quote_currency": cur,
                "rate": float(rate_str),
                "rate_date": pd.to_datetime(time_attr).date(),
                "retrieved_at": datetime.utcnow().isoformat(),
                "raw_hash": raw_hash,
                "raw_payload": ecb_data,
                "ingest_job_id": job_id,
                "created_at": datetime.utcnow().isoformat()
            })
    return pd.DataFrame(rows)

df1 = parse_ecb_api(job_id)
print(df1.head())

                              record_id   source base_currency quote_currency  \
0  b2492972-f39f-433f-87cf-5ecbdcaa0d30  ECB_API           EUR            USD   
1  8daaf02b-7722-4f9c-b809-fcbb5c65c8a4  ECB_API           EUR            JPY   
2  2c8e4815-ef01-4454-a06f-047553290ac1  ECB_API           EUR            BGN   
3  44f9206b-c39e-4896-8607-b37491c7ad51  ECB_API           EUR            CZK   
4  729f1b2f-830d-4269-8939-a9e5180e0e2c  ECB_API           EUR            DKK   

       rate   rate_date                retrieved_at  \
0    1.1766  2025-09-15  2025-09-16T09:39:53.329788   
1  173.3800  2025-09-15  2025-09-16T09:39:53.329788   
2    1.9558  2025-09-15  2025-09-16T09:39:53.331150   
3   24.3270  2025-09-15  2025-09-16T09:39:53.331150   
4    7.4646  2025-09-15  2025-09-16T09:39:53.332149   

                                            raw_hash  \
0  9b5fbb8efc2cbe489fb1b2191ea36c09d0e10c9d4e1545...   
1  9b5fbb8efc2cbe489fb1b2191ea36c09d0e10c9d4e1545...   
2  9b5fbb8efc2

In [6]:
print(df[df['quote_currency']=='USD']['rate'])
print(df1[df1['quote_currency']=='USD']['rate'])
df_exch = df
df_ecb = df1
df_ecb.ingest_job_id

df_all = pd.concat([df_exch, df_ecb], ignore_index=True) if not (df_exch.empty and df_ecb.empty) else pd.DataFrame()
df_all.ingest_job_id

152    1.179148
Name: rate, dtype: float64
0    1.1766
Name: rate, dtype: float64


0      88ab7982-7c11-4cee-9647-5311748e5f84
1      88ab7982-7c11-4cee-9647-5311748e5f84
2      88ab7982-7c11-4cee-9647-5311748e5f84
3      88ab7982-7c11-4cee-9647-5311748e5f84
4      88ab7982-7c11-4cee-9647-5311748e5f84
                       ...                 
197    88ab7982-7c11-4cee-9647-5311748e5f84
198    88ab7982-7c11-4cee-9647-5311748e5f84
199    88ab7982-7c11-4cee-9647-5311748e5f84
200    88ab7982-7c11-4cee-9647-5311748e5f84
201    88ab7982-7c11-4cee-9647-5311748e5f84
Name: ingest_job_id, Length: 202, dtype: object

In [ ]:


def to_numeric_bq(x):
    if pd.isnull(x):
        return None
    d = Decimal(str(x))
    # Round to 9 decimal places, truncate extra precision
    return d.quantize(Decimal("0.000000001"), rounding=ROUND_DOWN)

def write_to_bq(df: pd.DataFrame, client: bigquery.Client, gcp_project: str, bq_dataset: str, bq_landing_table: str, job_id: str):
    if df.empty:
        print("No data to write to BigQuery.")
        return

    landing_table_id = f"{gcp_project}.{bq_dataset}.{bq_landing_table}"

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND",
        schema=[
            bigquery.SchemaField("record_id", "STRING"),
            bigquery.SchemaField("source", "STRING"),
            bigquery.SchemaField("base_currency", "STRING"),
            bigquery.SchemaField("quote_currency", "STRING"),
            bigquery.SchemaField("rate", "NUMERIC"),
            bigquery.SchemaField("rate_date", "DATE"),
            bigquery.SchemaField("retrieved_at", "TIMESTAMP"),
            bigquery.SchemaField("raw_hash", "STRING"),
            bigquery.SchemaField("raw_payload", "STRING"),
            bigquery.SchemaField("ingest_job_id", "STRING"),
            bigquery.SchemaField("created_at", "TIMESTAMP"),
        ],
    )

    df_load = df[["record_id","source","base_currency","quote_currency","rate",
        "rate_date","retrieved_at","raw_hash","raw_payload",
        "ingest_job_id","created_at"
    ]].copy()

    # Convert columns to correct types
    df_load["rate_date"] = pd.to_datetime(df_load["rate_date"], errors="coerce").dt.date
    df_load["retrieved_at"] = pd.to_datetime(df_load["retrieved_at"])
    df_load["created_at"] = pd.to_datetime(df_load["created_at"])
    df_load["rate"] = df_load["rate"].apply(to_numeric_bq)   


    # Load to BigQuery
    load_job = client.load_table_from_dataframe(df_load, landing_table_id, job_config=job_config)
    load_job.result()
    print(f"✅ Loaded {load_job.output_rows} rows into {landing_table_id}")
    logger.info("Successfully loaded %s record into landing table %s", load_job.output_rows, landing_table_id)


    finaltable_id = f"{gcp_project}.{bq_dataset}.{bq_final_table}"

    final_table_query = f"""
    MERGE INTO `{finaltable_id}` t
    USING
    (
        WITH
            exchangerates_api AS 
            (
                SELECT
                *
                FROM `{landing_table_id}`
                WHERE source='ExchangeRates_API'
            ),
            ecb_api AS 
            (
                SELECT
                *
                FROM `{landing_table_id}`
                WHERE source='ECB_API'
            )
        SELECT
            ecb_api.record_id AS ecb_api_record_id,
            exchangerates_api.record_id AS exchangerates_api_record_id,
            IFNULL(ecb_api.base_currency, exchangerates_api.base_currency) AS base_currency,
            IFNULL(ecb_api.quote_currency, exchangerates_api.quote_currency) AS quote_currency,
            IFNULL(ecb_api.rate_date, exchangerates_api.rate_date) AS rate_date,
            ecb_api.rate AS ecb_api_rate,
            exchangerates_api.rate AS exchangerates_api_rates,
            ecb_api.retrieved_at AS ecb_api_retrieved_at,
            exchangerates_api.retrieved_at AS exchangerates_api_retrieved_at,
            IFNULL(ecb_api.ingest_job_id, exchangerates_api.ingest_job_id) AS ingest_job_id,
            CURRENT_TIMESTAMP() AS created_at
        FROM ecb_api
        FULL JOIN
            exchangerates_api
        ON
            ecb_api.ingest_job_id = exchangerates_api.ingest_job_id
            AND ecb_api.base_currency = exchangerates_api.base_currency
            AND ecb_api.quote_currency = exchangerates_api.quote_currency) s
    ON
        s.base_currency = t.base_currency
        AND s.quote_currency = t.quote_currency
        AND s.rate_date = t.rate_date
    WHEN MATCHED AND 
        (s.ecb_api_rate <> t.ecb_api_rate OR s.exchangerates_api_rates <> t.exchangerates_api_rates) 
    THEN UPDATE 
        SET 
        t.ecb_api_record_id=s.ecb_api_record_id, 
        t.exchangerates_api_record_id=s.exchangerates_api_record_id, 
        t.ecb_api_rate=s.ecb_api_rate, 
        t.exchangerates_api_rates=s.exchangerates_api_rates, 
        t.ecb_api_retrieved_at=s.ecb_api_retrieved_at, 
        t.exchangerates_api_retrieved_at = s.exchangerates_api_retrieved_at, 
        t.ingest_job_id = s.ingest_job_id, 
        t.updated_at = CURRENT_TIMESTAMP()
    WHEN NOT MATCHED
    THEN INSERT
    (
        ecb_api_record_id,
        exchangerates_api_record_id,
        base_currency,
        quote_currency,
        rate_date,
        ecb_api_rate,
        exchangerates_api_rates,
        ecb_api_retrieved_at,
        exchangerates_api_retrieved_at,
        ingest_job_id,
        created_at 
    )
    VALUES
    (
        ecb_api_record_id, 
        exchangerates_api_record_id, 
        base_currency, quote_currency, 
        rate_date, ecb_api_rate, 
        exchangerates_api_rates, 
        ecb_api_retrieved_at, 
        exchangerates_api_retrieved_at, 
        ingest_job_id, 
        created_at 
    )                        
    """

    query_job = client.query(final_table_query)
    query_job.result()

    print(f"✅ Merged {query_job.num_dml_affected_rows}  rows into {finaltable_id}")
    logger.info("Merge job successfully merged %s record into final table %s", query_job.num_dml_affected_rows, finaltable_id)


df_all = pd.concat([df_exch, df_ecb], ignore_index=True) if not (df_exch.empty and df_ecb.empty) else pd.DataFrame()

write_to_bq(df_all, bigquery.Client(), gcp_project, bq_dataset, bq_landing_table, job_id)


C:\Users\sudyr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Data types before loading to BigQuery:
record_id                 object
source                    object
base_currency             object
quote_currency            object
rate                      object
rate_date                 object
retrieved_at      datetime64[ns]
raw_hash                  object
raw_payload               object
ingest_job_id             object
created_at        datetime64[ns]
dtype: object


C:\Users\sudyr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\_pandas_helpers.py:484: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ Loaded 202 rows into dataengineering-466011.lexis_nexis_task.exchange_rates_landing
✅ Merged 0  rows into dataengineering-466011.lexis_nexis_task.exchange_rates


In [ ]:
def validate_exchange_rates_data(df: pd.DataFrame) -> pd.DataFrame:

    currency_columns = ["base_currency", "quote_currency"]

    df = df.dropna() # Drop rows with any NULLs

    # Filter valid currency codes
    currency_columns = ["base_currency", "quote_currency"]
    mask = df[currency_columns].astype(str).apply(lambda col: col.str.match(r"^[A-Z]{3}$")).any(axis=1)
    df = df[mask] 

    df = df[df["rate"].apply(lambda x: isinstance(x, (int, float)) and x > 0 and x < 1000)] # Filter valid rates

    df = df.drop_duplicates(keep="last") # Deduplicate

    return df

In [10]:
validate_exchange_rates_data(df_all).head()

AttributeError: 'DataFrame' object has no attribute 'str'

In [14]:
currency_columns = ["base_currency", "quote_currency"]
mask = df_all[currency_columns].astype(str).apply(lambda col: col.str.match(r"^[A-Z]{3}$")).any(axis=1)
df = df[mask]
df.head()

C:\Users\sudyr\AppData\Local\Temp\ipykernel_11340\751206873.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[mask]


,record_id,source,base_currency,quote_currency,rate,rate_date,retrieved_at,raw_hash,raw_payload,ingest_job_id,created_at
0,81eaa265-209e-4488-9b58-903b760748e6,ExchangeRates_API,EUR,AED,4.330420,2025-09-16,2025-09-16T09:39:49.357727,62dbb14a476a25b50243bc633f4406b9d13a2525d897c0...,"{""base"": ""EUR"", ""date"": ""2025-09-16"", ""rates"":...",88ab7982-7c11-4cee-9647-5311748e5f84,2025-09-16T09:39:49.357727
1,d4bbc144-fb69-48ed-aa82-f06cd6a8536c,ExchangeRates_API,EUR,AFN,81.360861,2025-09-16,2025-09-16T09:39:49.357727,62dbb14a476a25b50243bc633f4406b9d13a2525d897c0...,"{""base"": ""EUR"", ""date"": ""2025-09-16"", ""rates"":...",88ab7982-7c11-4cee-9647-5311748e5f84,2025-09-16T09:39:49.357727
2,ffc86fa7-2321-49ef-a5f1-28e136a8c1ad,ExchangeRates_API,EUR,ALL,97.165585,2025-09-16,2025-09-16T09:39:49.357727,62dbb14a476a25b50243bc633f4406b9d13a2525d897c0...,"{""base"": ""EUR"", ""date"": ""2025-09-16"", ""rates"":...",88ab7982-7c11-4cee-9647-5311748e5f84,2025-09-16T09:39:49.357727
3,853609e5-9dc0-4f0d-93a2-c7a20bb684c1,ExchangeRates_API,EUR,AMD,451.754905,2025-09-16,2025-09-16T09:39:49.357727,62dbb14a476a25b50243bc633f4406b9d13a2525d897c0...,"{""base"": ""EUR"", ""date"": ""2025-09-16"", ""rates"":...",88ab7982-7c11-4cee-9647-5311748e5f84,2025-09-16T09:39:49.357727
4,35baf66c-ba11-44f9-99bd-c48ffa5db406,ExchangeRates_API,EUR,ANG,2.111150,2025-09-16,2025-09-16T09:39:49.357727,62dbb14a476a25b50243bc633f4406b9d13a2525d897c0...,"{""base"": ""EUR"", ""date"": ""2025-09-16"", ""rates"":...",88ab7982-7c11-4cee-9647-5311748e5f84,2025-09-16T09:39:49.357727
